# 01- Experiment Objective

# 02 Experimental Definition

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.core.utility_experiment_config import UtilityExperimentConfig
from src.core.dataset_config import DatasetConfig
from src.core.preprocessing_config import PreprocessingConfig
from src.core.task_config import TaskConfig


In [ ]:
CATEGORICAL_COLUMNS= [
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q018",
    "Q021",
    "Q022",
]

NUMERICAL_COLUMNS= [
    "TP_FAIXA_ETARIA",
    "TP_ANO_CONCLUIU",
]

# Run the experiment where both features and targets are affected by differential privacy.

In [ ]:
def get_fully_private_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-07-21_23-19-59",
        data_sample_size=100_000,
        data_random_state=42,
    )

    tasks_config = [
        TaskConfig(task_type="classification", target="Q007"),
        TaskConfig(task_type="regression", target="TP_FAIXA_ETARIA"),
    ]

    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS,
        numerical_columns=NUMERICAL_COLUMNS,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        tasks=tasks_config,
        preprocessing=preprocessing_config,
    )

# Run the experiment where only the features are affected by differential privacy, while the targets remain unchanged.

In [ ]:
def get_only_private_features_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-07-21_23-19-59",
        data_sample_size=100_000,
        data_random_state=42,
    )
    
    tasks_config = [
        TaskConfig(task_type="classification", target="TP_SEXO"),
        TaskConfig(task_type="regression", target="Q005"),
    ]

    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS,
        numerical_columns=NUMERICAL_COLUMNS,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        tasks=tasks_config,
        preprocessing=preprocessing_config,
    )

# Select experiment

In [ ]:
evaluation_config = get_fully_private_config()

03 Dataset Preparation

In [ ]:
from src.data.dataset_registry import load_dataset_bundle
from src.data.sample_dataset import sample_dataset_bundle

In [ ]:
dataset_bundle = load_dataset_bundle(
    dataset_name=evaluation_config.dataset.dataset_name,
    dataset_version=evaluation_config.dataset.dataset_version,
)

dataset_sample = sample_dataset_bundle(
    dataset_bundle,
    sample_size=evaluation_config.dataset.data_sample_size,
    random_state=evaluation_config.dataset.data_random_state,
)

04 Feature Preparation

In [ ]:
from src.core.splits_config import SplitConfig

split_plan = SplitConfig(seed=42, test_size=0.5)

In [ ]:
from src.experiments.utility_evaluation_services import feature_preparation

prepared_features = []

for dataset_name, df in zip(dataset_bundle['dataset_names'], dataset_bundle['datasets']):
    for task in evaluation_config.tasks:
        prepared = feature_preparation.prepare_features(
            name=dataset_name,
            df=df,
            task_config=task,
            split_plan=split_plan,
            preprocessing_config=evaluation_config.preprocessing,
        )
        prepared_features.append(prepared)

05 Model execution

In [ ]:
from src.core.models_spec_config import ModelSpec

# Classification Models

In [ ]:
XGBOOST_CLASSIFIER = ModelSpec(
    name="xgboost",
    model_type="xgboost_classifier",
    parameters={
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)

RANDOM_FOREST_CLASSIFIER = ModelSpec(
    name="random_forest",
    model_type="random_forest_classifier",
    parameters={
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 8,
        "min_samples_split": 10,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": 2,
    },
)

LOGISTIC_REGRESSION = ModelSpec(
    name="logistic regression",
    model_type="logistic_regression",
    parameters={"max_iter": 5000, "solver": "saga"},
)

# Regression Models

In [ ]:
XGBOOST_REGRESSOR = ModelSpec(
    name="xgboost",
    model_type="xgboost_regressor",
    parameters={
        "objective": "reg:squarederror",
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)

RANDOM_FOREST_REGRESSOR = ModelSpec(
    name="random_forest",
    model_type="random_forest_regressor",
    parameters={
        "n_estimators": 100,
        "max_depth": 8,
        "min_samples_leaf": 8,
        "min_samples_split": 10,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": 2,
    },
)

LINEAR_REGRESSION = ModelSpec(
    name="linear regression",
    model_type="linear_regression",
    parameters={},
)

In [ ]:
CLASSIFICATION_MODELS = [
    XGBOOST_CLASSIFIER,
    RANDOM_FOREST_CLASSIFIER,
    LOGISTIC_REGRESSION,
]


REGRESSION_MODELS = [
    XGBOOST_REGRESSOR,
    RANDOM_FOREST_REGRESSOR,
    LINEAR_REGRESSION,
]

# Running models

In [ ]:
from src.experiments.utility_evaluation_services.model import model_runner

06 Utility Evaluation

In [ ]:
from src.experiments.utility_evaluation_services import metrics

from src.core.results_config import UtilityClassificationResult, UtilityRegressionResult


In [ ]:
utility_records = []
leakage_input= []

for prepared in prepared_features:

    models = (
        CLASSIFICATION_MODELS
        if prepared.task_type == "classification"
        else REGRESSION_MODELS
    )

    for model_spec in models:

        prediction = model_runner.execute_model(
            prepared_features=prepared,
            model_spec=model_spec,
        )

        utility = metrics.compute_utility_metrics(
            prediction_result=prediction,
            task_type=prepared.task_type,
        )

        record  = {
            "dataset": prepared.name,
            "task_type": prepared.task_type,
            "target": prepared.target,
            "model": model_spec.name,
            "model_type": model_spec.model_type,
        }

        if type(utility)  == UtilityClassificationResult:
            record.update({
                "test_acc": utility.test_acc,
                "train_acc": utility.train_acc,
                "validation_acc": utility.validation_acc,
                "test_precision": utility.test_precision,
                "test_recall": utility.test_recall,
                "test_f1": utility.test_f1,
                "generalization_gap_%": utility.generalization_gap,
            })


            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "train_proba": prediction.train_proba,
                    "test_proba": prediction.test_proba,
                    "y_train_encoded": prediction.y_train_encoded,
                    "y_test_encoded": prediction.y_test_encoded,
                },
    })

            
        elif type(utility) == UtilityRegressionResult:
            record.update({
                "test_mae": utility.test_mae,
                "test_r2_score": utility.test_r2,
                "train_r2_score": utility.train_r2,
                "validation_r2_score": utility.validation_r2,
                "generalization_gap_%": utility.generalization_gap,
            })

            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "y_train_true": prediction.y_train_true,
                    "y_train_pred": prediction.y_train_pred,
                    "y_test_true": prediction.y_test_true,
                    "y_test_pred": prediction.y_test_pred,
                },
            })

        utility_records.append(record)

        


07 Visualization

08 Export

In [ ]:
import pandas as pd
from datetime import datetime

from artifacts.persistence import persist_utility_artifact

EXPERIMENT_TYPE = "utility_evaluation"
ARTIFACT_SCHEMA_VERSION = "1.0"

experiment_id =  datetime.now().strftime("%Y%m%d_%H%M%S")

utility_metrics = pd.DataFrame(utility_records)
utility_metrics.insert(0, "experiment_id", experiment_id)

input_leakage= pd.DataFrame(leakage_input)

model_specs = {
    (model.name, model.model_type): {
        "name": model.name,
        "model_type": model.model_type,
        "parameters": model.parameters,
    }
    for model in [*CLASSIFICATION_MODELS, *REGRESSION_MODELS]
}

experiment_metadata = {
    "experiment_id": experiment_id,
    "experiment_type": EXPERIMENT_TYPE,
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": {
        "name": evaluation_config.dataset.dataset_name,
        "version": evaluation_config.dataset.dataset_version,
        "sample_size": evaluation_config.dataset.data_sample_size,
        "random_state": evaluation_config.dataset.data_random_state,
    },
    "split": {
        "seed": split_plan.seed,
        "test_size": split_plan.test_size,
    },
    "preprocessing": {
        "categorical_columns": evaluation_config.preprocessing.categorical_columns,
        "numerical_columns": evaluation_config.preprocessing.numerical_columns,
    },
    "tasks": [
        {"task_type": task.task_type, "target": task.target}
        for task in evaluation_config.tasks
    ],
    "models": list(model_specs.values()),
}

artifact_path = persist_utility_artifact(
    experiment_id=experiment_id,
    metadata=experiment_metadata,
    utility_metrics=utility_metrics,
    input_leakage= input_leakage
)

artifact_path

10 Conclusion / Observations